In [1]:
"""
Generate Agent Messages for Fusion Training

This notebook:
1. Loads all trained agent models
2. Generates predictions on validation data
3. Creates structured AgentMessage objects
4. Prepares fusion training data
"""

import sys
sys.path.append('../../')  

import pandas as pd
import numpy as np
import joblib
import pickle
from pathlib import Path
from tqdm import tqdm

# Import communication protocol
from src.communication.protocol import AgentMessage, DISEASE_LIST, create_fusion_features

print("✅ Imports complete")
print(f"Disease list: {DISEASE_LIST}")

✅ Imports complete
Disease list: ['SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE', 'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE', 'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE', 'ANEMIA', 'PANCREATITIS']


In [7]:
print("Loading all agent models...\n")

# Agent 1 (Labs) - XGBoost models
print("Loading Agent 1 (Labs)...")
agent1_models = {}
for disease in DISEASE_LIST:
    disease_filename = disease.lower()
    model_path = f'../../models/agent1_{disease_filename}.joblib'
    agent1_models[disease] = joblib.load(model_path)
print(f"  ✅ Loaded {len(agent1_models)} Agent 1 models")

# Agent 2 (Notes) - Using pre-generated predictions from Colab
print("Agent 2 (Notes)...")
print("  ✅ Will use pre-generated predictions from Colab")
agent2_models = None 

# Agent 3 (Vitals) - XGBoost models
print("Loading Agent 3 (Vitals)...")
agent3_models = {}
for disease in DISEASE_LIST:
    disease_filename = disease.lower()
    model_path = f'../../models/agent3_{disease_filename}.joblib'
    agent3_models[disease] = joblib.load(model_path)
print(f"  ✅ Loaded {len(agent3_models)} Agent 3 models")

print("\n✅ Agent 1 and Agent 3 models loaded!")
print("✅ Agent 2 predictions ready from Colab!")

Loading all agent models...

Loading Agent 1 (Labs)...
  ✅ Loaded 9 Agent 1 models
Agent 2 (Notes)...
  ✅ Will use pre-generated predictions from Colab
Loading Agent 3 (Vitals)...
  ✅ Loaded 9 Agent 3 models

✅ Agent 1 and Agent 3 models loaded!
✅ Agent 2 predictions ready from Colab!


In [8]:
print("Loading validation data...\n")

# Load processed features
labs_val = pd.read_csv('../../data/processed/lab_features_val.csv')
print(f"  Labs: {labs_val.shape}")

notes_val = pd.read_csv('../../data/processed/notes_val.csv')  
print(f"  Notes: {notes_val.shape}")

vitals_val = pd.read_csv('../../data/processed/vital_features_val.csv')
print(f"  Vitals: {vitals_val.shape}")

# Load labels
labels_df = pd.read_csv('../../data/processed/patient_multilabels.csv')

# Get validation patient IDs
val_patient_ids = labs_val['SUBJECT_ID'].values
val_labels = labels_df[labels_df['SUBJECT_ID'].isin(val_patient_ids)].copy()

print(f"\n✅ Validation set: {len(val_labels)} patients")

Loading validation data...

  Labs: (4675, 156)
  Notes: (4643, 2)
  Vitals: (4675, 66)

✅ Validation set: 4675 patients


In [13]:
print("Generating Agent 1 (Labs) messages...\n")

# ============================================================
# LOAD PREPROCESSING OBJECTS
# ============================================================
print("Loading Agent 1 preprocessing objects...")

# Load imputer and scaler
imputer = joblib.load('../../models/agent1_imputer.joblib')
scaler = joblib.load('../../models/agent1_scaler.joblib')

print("  ✅ Loaded imputer and scaler")

# Load preprocessing info
import json
with open('../../models/agent1_preprocessing_info.json', 'r') as f:
    preprocessing_info = json.load(f)

training_features = preprocessing_info['features_used']  # ← Fixed key name!
features_dropped = preprocessing_info['features_dropped']

print(f"  ✅ Training used {len(training_features)} features")
print(f"  ✅ Dropped {len(features_dropped)} features during training")

# ============================================================
# PREPROCESS VALIDATION DATA
# ============================================================
print("\nPreprocessing validation data...")

# Get feature columns from validation data (drop SUBJECT_ID)
val_feature_cols = [col for col in labs_val.columns if col != 'SUBJECT_ID']
print(f"  Raw validation features: {len(val_feature_cols)}")

# Check which training features are available in validation data
missing_features = set(training_features) - set(val_feature_cols)
if missing_features:
    print(f"  ⚠️  Missing features in validation: {len(missing_features)}")
    raise ValueError(f"Validation data is missing features that were in training: {list(missing_features)[:5]}...")

# Select only the features used in training (in correct order)
X_val_raw = labs_val[training_features].values

print(f"  ✅ Aligned to {X_val_raw.shape[1]} features (matching training)")

# Apply imputer (fill missing values using median strategy)
print(f"  Imputing missing values (strategy: {preprocessing_info['imputation_strategy']})...")
X_val_imputed = imputer.transform(X_val_raw)

# Apply scaler (standardize using StandardScaler)
print(f"  Scaling features (method: {preprocessing_info['scaling_method']})...")
X_val_scaled = scaler.transform(X_val_imputed)

print(f"  ✅ Preprocessed validation data: {X_val_scaled.shape}")

# ============================================================
# GENERATE MESSAGES
# ============================================================
print("\nGenerating Agent 1 messages...")

agent1_messages = []

for idx, patient_id in enumerate(tqdm(labs_val['SUBJECT_ID'], desc="Agent 1")):
    # Get preprocessed features for this patient
    X = X_val_scaled[idx:idx+1]  # Shape: (1, 143)
    
    # Get predictions for all 9 diseases
    probabilities = {}
    for disease in DISEASE_LIST:
        prob = agent1_models[disease].predict_proba(X)[0, 1]
        probabilities[disease] = float(prob)
    
    # Get top prediction
    top_disease = max(probabilities, key=probabilities.get)
    top_confidence = probabilities[top_disease]
    
    # Create message
    message = AgentMessage(
        agent_name='agent1_labs',
        prediction=top_disease,
        confidence=top_confidence,
        probabilities=probabilities,
        top_features=[],
        metadata={
            'data_available': True,
            'model_type': 'xgboost',
            'preprocessed': True,
            'n_features': len(training_features)
        }
    )
    
    agent1_messages.append({
        'SUBJECT_ID': patient_id,
        'agent': 'agent1',
        'message': message.to_dict()
    })

print(f"\n✅ Generated {len(agent1_messages)} Agent 1 messages")

C:\Users\athar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
C:\Users\athar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Generating Agent 1 (Labs) messages...

Loading Agent 1 preprocessing objects...
  ✅ Loaded imputer and scaler
  ✅ Training used 143 features
  ✅ Dropped 12 features during training

Preprocessing validation data...
  Raw validation features: 155
  ✅ Aligned to 143 features (matching training)
  Imputing missing values (strategy: median)...
  Scaling features (method: StandardScaler)...
  ✅ Preprocessed validation data: (4675, 143)

Generating Agent 1 messages...


Agent 1: 100%|████████████████████████████████████| 4675/4675 [00:50<00:00, 93.15it/s]


✅ Generated 4675 Agent 1 messages


In [14]:
print("Generating Agent 2 (Notes) messages...\n")

predictions_file = '../../results/agent2_validation_predictions.csv'

if not Path(predictions_file).exists():
    raise FileNotFoundError(
        f"❌ Agent 2 predictions not found!\n"
        f"   Expected: {predictions_file}\n"
        f"   Please place the downloaded CSV in mimic_project/results/"
    )

agent2_predictions = pd.read_csv(predictions_file)
print(f"✅ Loaded Agent 2 predictions: {agent2_predictions.shape}")

# Verify all diseases are present
missing_diseases = [d for d in DISEASE_LIST if d not in agent2_predictions.columns]
if missing_diseases:
    raise ValueError(f"Missing diseases in predictions: {missing_diseases}")

agent2_messages = []

for idx, row in tqdm(agent2_predictions.iterrows(), 
                     total=len(agent2_predictions), 
                     desc="Agent 2"):
    patient_id = row['SUBJECT_ID']
    
    # Get probabilities for all 9 diseases
    probabilities = {disease: float(row[disease]) for disease in DISEASE_LIST}
    
    # Get top prediction
    top_disease = max(probabilities, key=probabilities.get)
    top_confidence = probabilities[top_disease]
    
    # Create message
    message = AgentMessage(
        agent_name='agent2_notes',
        prediction=top_disease,
        confidence=top_confidence,
        probabilities=probabilities,
        top_features=[],
        metadata={
            'data_available': True,
            'model_type': 'bert',
            'source': 'colab_trained_models'
        }
    )
    
    agent2_messages.append({
        'SUBJECT_ID': patient_id,
        'agent': 'agent2',
        'message': message.to_dict()
    })

print(f"✅ Generated {len(agent2_messages)} Agent 2 messages from REAL BERT predictions")

Generating Agent 2 (Notes) messages...

✅ Loaded Agent 2 predictions: (4643, 10)


Agent 2: 100%|██████████████████████████████████| 4643/4643 [00:00<00:00, 7750.18it/s]

✅ Generated 4643 Agent 2 messages from REAL BERT predictions


In [18]:
print("Generating Agent 3 (Vitals) messages...\n")

# ============================================================
# LOAD PREPROCESSING OBJECTS
# ============================================================
print("Loading Agent 3 preprocessing objects...")

# Load imputer and scaler
imputer_agent3 = joblib.load('../../models/agent3_imputer.joblib')
scaler_agent3 = joblib.load('../../models/agent3_scaler.joblib')

print("  ✅ Loaded imputer and scaler")

# Load preprocessing info
with open('../../models/agent3_preprocessing_info.json', 'r') as f:
    preprocessing_info_agent3 = json.load(f)

training_features_agent3 = preprocessing_info_agent3['features_used']
features_dropped_agent3 = preprocessing_info_agent3['features_dropped']

print(f"  ✅ Training used {len(training_features_agent3)} features")
print(f"  ✅ Dropped {len(features_dropped_agent3)} features during training")

# ============================================================
# CREATE MISSING FEATURES
# ============================================================
print("\nChecking for missing features...")

val_feature_cols_agent3 = [col for col in vitals_val.columns if col != 'SUBJECT_ID']
missing_features_agent3 = set(training_features_agent3) - set(val_feature_cols_agent3)

if missing_features_agent3:
    print(f"  ⚠️  Missing features: {missing_features_agent3}")
    
    # Handle NUM_DISEASES if it's missing
    if 'NUM_DISEASES' in missing_features_agent3:
        print("  Creating NUM_DISEASES feature...")
        
        # Count number of diseases per patient from labels
        # Merge vitals_val with val_labels to get disease counts
        vitals_with_labels = vitals_val.merge(
            val_labels[['SUBJECT_ID'] + DISEASE_LIST], 
            on='SUBJECT_ID', 
            how='left'
        )
        
        # Count how many diseases each patient has
        vitals_with_labels['NUM_DISEASES'] = vitals_with_labels[DISEASE_LIST].sum(axis=1)
        
        # Update vitals_val with the new feature
        vitals_val['NUM_DISEASES'] = vitals_with_labels['NUM_DISEASES']
        
        print(f"    ✅ Created NUM_DISEASES (mean: {vitals_val['NUM_DISEASES'].mean():.2f})")
        
        # Remove from missing features list
        missing_features_agent3.remove('NUM_DISEASES')
    
    # Check if there are still missing features
    if missing_features_agent3:
        raise ValueError(f"Still missing features: {list(missing_features_agent3)}")

# ============================================================
# PREPROCESS VALIDATION DATA
# ============================================================
print("\nPreprocessing validation data...")

val_feature_cols_agent3 = [col for col in vitals_val.columns if col != 'SUBJECT_ID']
print(f"  Total validation features: {len(val_feature_cols_agent3)}")

# Align features 
X_val_raw_agent3 = vitals_val[training_features_agent3].values
print(f"  ✅ Aligned to {X_val_raw_agent3.shape[1]} features (matching training)")

# Apply preprocessing
print(f"  Imputing missing values (strategy: {preprocessing_info_agent3['imputation_strategy']})...")
X_val_imputed_agent3 = imputer_agent3.transform(X_val_raw_agent3)

print(f"  Scaling features (method: {preprocessing_info_agent3['scaling_method']})...")
X_val_scaled_agent3 = scaler_agent3.transform(X_val_imputed_agent3)

print(f"  ✅ Preprocessed validation data: {X_val_scaled_agent3.shape}")

# ============================================================
# GENERATE MESSAGES
# ============================================================
print("\nGenerating Agent 3 messages...")

agent3_messages = []

for idx, patient_id in enumerate(tqdm(vitals_val['SUBJECT_ID'], desc="Agent 3")):
    # Get preprocessed features
    X = X_val_scaled_agent3[idx:idx+1]
    
    # Get predictions for all 9 diseases
    probabilities = {}
    for disease in DISEASE_LIST:
        prob = agent3_models[disease].predict_proba(X)[0, 1]
        probabilities[disease] = float(prob)
    
    # Get top prediction
    top_disease = max(probabilities, key=probabilities.get)
    top_confidence = probabilities[top_disease]
    
    # Create message
    message = AgentMessage(
        agent_name='agent3_vitals',
        prediction=top_disease,
        confidence=top_confidence,
        probabilities=probabilities,
        top_features=[],
        metadata={
            'data_available': True,
            'model_type': 'xgboost',
            'preprocessed': True,
            'n_features': len(training_features_agent3)
        }
    )
    
    agent3_messages.append({
        'SUBJECT_ID': patient_id,
        'agent': 'agent3',
        'message': message.to_dict()
    })

print(f"\n✅ Generated {len(agent3_messages)} Agent 3 messages")



C:\Users\athar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
C:\Users\athar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Generating Agent 3 (Vitals) messages...

Loading Agent 3 preprocessing objects...
  ✅ Loaded imputer and scaler
  ✅ Training used 62 features
  ✅ Dropped 4 features during training

Checking for missing features...
  ⚠️  Missing features: {'NUM_DISEASES'}
  Creating NUM_DISEASES feature...
    ✅ Created NUM_DISEASES (mean: 2.68)

Preprocessing validation data...
  Total validation features: 66
  ✅ Aligned to 62 features (matching training)
  Imputing missing values (strategy: median)...
  Scaling features (method: StandardScaler)...
  ✅ Preprocessed validation data: (4675, 62)

Generating Agent 3 messages...


Agent 3: 100%|███████████████████████████████████| 4675/4675 [00:43<00:00, 107.82it/s]


✅ Generated 4675 Agent 3 messages


In [21]:
print("Converting messages to feature matrix...\n")

# Import the helper function from protocol
from src.communication.protocol import create_fusion_features

X_fusion = []
y_fusion = {disease: [] for disease in DISEASE_LIST}

print("Creating feature vectors for each patient...")

for entry in tqdm(fusion_data, desc="Creating features"):
    # Create feature vector from three agent messages
    features = create_fusion_features(
        entry['agent1_message'],
        entry['agent2_message'],
        entry['agent3_message']
    )
    
    X_fusion.append(features)
    
    # Add labels for each disease
    for i, disease in enumerate(DISEASE_LIST):
        y_fusion[disease].append(entry['true_labels'][i])

# Convert to numpy arrays
X_fusion = np.array(X_fusion)

print(f"\n✅ Fusion feature matrix created!")
print(f"   Shape: {X_fusion.shape}")
print(f"   Features breakdown:")
print(f"     - Agent 1 probabilities: 9 features (indices 0-8)")
print(f"     - Agent 2 probabilities: 9 features (indices 9-17)")
print(f"     - Agent 3 probabilities: 9 features (indices 18-26)")
print(f"     - Agent confidences: 3 features (indices 27-29)")
print(f"     - Data availability flags: 3 features (indices 30-32)")
print(f"   Total: 33 features per patient")

# ============================================================
# SAVE FUSION FEATURES
# ============================================================
print("\n💾 Saving fusion features...")

np.save('../../data/processed/X_fusion_val.npy', X_fusion)
print("  ✅ Saved: X_fusion_val.npy")

# Save labels for each disease
for disease in DISEASE_LIST:
    y_array = np.array(y_fusion[disease])
    disease_filename = disease.lower()
    np.save(f'../../data/processed/y_fusion_val_{disease_filename}.npy', y_array)

print(f"  ✅ Saved {len(DISEASE_LIST)} label files")

# ============================================================
# QUICK STATISTICS
# ============================================================
print("\n📊 Feature Statistics:")

# Show feature ranges
feature_mins = X_fusion.min(axis=0)
feature_maxs = X_fusion.max(axis=0)
feature_means = X_fusion.mean(axis=0)

print("\n  Agent 1 probabilities (first 3 diseases):")
for i in range(3):
    disease = DISEASE_LIST[i]
    print(f"    {disease:30s} Mean: {feature_means[i]:.3f}, Range: [{feature_mins[i]:.3f}, {feature_maxs[i]:.3f}]")

print("\n  Agent confidences:")
print(f"    Agent 1: {feature_means[27]:.3f}")
print(f"    Agent 2: {feature_means[28]:.3f}")
print(f"    Agent 3: {feature_means[29]:.3f}")

print("\n  Data availability flags (should all be 1.0):")
print(f"    Agent 1: {feature_means[30]:.1f}")
print(f"    Agent 2: {feature_means[31]:.1f}")
print(f"    Agent 3: {feature_means[32]:.1f}")

print("\n✅ Fusion features ready for training!")

Converting messages to feature matrix...

Creating feature vectors for each patient...


Creating features: 100%|███████████████████████| 4643/4643 [00:00<00:00, 46388.35it/s]


✅ Fusion feature matrix created!
   Shape: (4643, 33)
   Features breakdown:
     - Agent 1 probabilities: 9 features (indices 0-8)
     - Agent 2 probabilities: 9 features (indices 9-17)
     - Agent 3 probabilities: 9 features (indices 18-26)
     - Agent confidences: 3 features (indices 27-29)
     - Data availability flags: 3 features (indices 30-32)
   Total: 33 features per patient

💾 Saving fusion features...
  ✅ Saved: X_fusion_val.npy
  ✅ Saved 9 label files

📊 Feature Statistics:

  Agent 1 probabilities (first 3 diseases):
    SEPSIS                         Mean: 0.180, Range: [0.001, 0.956]
    PNEUMONIA                      Mean: 0.202, Range: [0.003, 0.938]
    RESPIRATORY_FAILURE            Mean: 0.362, Range: [0.016, 0.995]

  Agent confidences:
    Agent 1: 0.753
    Agent 2: 0.791
    Agent 3: 0.566

  Data availability flags (should all be 1.0):
    Agent 1: 1.0
    Agent 2: 1.0
    Agent 3: 1.0

✅ Fusion features ready for training!
